# 4. Dictionaries


## 25: Be Cautious when Relying on Dictionary Insertion Ordering

### Notatka: **kwargs

- W sygnaturze funkcji zapis `**kwargs` zbiera wszystkie dodatkowe argumenty nazwane i umieszcza je w słowniku (dict) o nazwie `kwargs`.
- Do elementów można się odwoływać jak do zwykłego słownika: `kwargs['klucz']` lub iterując: `for k, v in kwargs.items(): ...`.
- Przykład:

```python
def my_func(**kwargs):
    for key, value in kwargs.items():
        print(f"{key} = {value}")

my_func(goose='gosling', kangaroo='joey')
```

- Operator `**` przy wywołaniu funkcji rozpakowuje słownik na argumenty nazwane: `f(**{'x':1, 'y':2})`.
- `kwargs` to zwykły dict; od Pythona 3.7 zachowuje kolejność wstawiania.
- Zastosowania: przekazywanie opcjonalnych parametrów, przekazywanie dalej (forwarding), elastyczne API.

### Notatki: dict vs SortedDict

- **dict (wbudowany)**
  - Przechowuje pary klucz→wartość; od Pythona 3.7 zachowuje kolejność wstawiania (insertion order).
  - Operacje: odczyt/wstawienie/usunięcie ~ O(1) średnio.
  - Metody: keys(), values(), items(), popitem(), get(), update() itp.
  - Zastosowanie: szybki dostęp po kluczu; gdy ważna jest kolejność wstawiania.

- **SortedDict (posortowany)**
  - Ogólne pojęcie: mapowanie, które iteruje w kolejności posortowanej według klucza.
  - Implementacje różne (własne klasy sortujące klucze, sortedcontainers.SortedDict, bisect+listy): różne złożoności.
  - Typowe złożoności: wstawianie/odczyt O(log n) dla struktur drzewiastych; iteracja w porządku posortowanym O(n).
  - Zastosowanie: gdy potrzebna jest zawsze posortowana iteracja lub zapytania zakresowe.

- **Główne różnice**
  - Kolejność: dict → insertion order; SortedDict → porządek posortowany według klucza.
  - Wydajność: dict jest szybszy dla operacji kluczowych; SortedDict ma większy koszt (implementacja zależna).
  - API/typ: SortedDict bywa Mappingiem, niekoniecznie instancją dict — unikaj sprawdzania isinstance(obj, dict), używaj duck-typing lub Mapping.
  - Pamięć: implementacje posortowane zwykle mają większy narzut pamięci.

- **Pułapki i wskazówki praktyczne**
  - Nie polegaj na next(iter(mapping)) jako „zwycięzcy” bez sprawdzenia typu — dla dict to element wg insertion order, dla SortedDict to najmniejszy (posortowany).
  - Jeśli iterujesz i klucze są sortowane przy każdej iteracji, pamiętaj o koszcie O(n log n).
  - Wybierz dict dla szybkości i prostoty; wybierz SortedDict (np. z paczki sortedcontainers) gdy potrzebujesz gwarantowanej posortowanej kolejności i operacji zakresowych.

- Jeśli chcesz, mogę dodać krótki benchmark porównawczy (dla Twoich danych) lub przykład implementacji SortedDict (bisect / sortedcontainers).

In [1]:
baby_names = {
    "cat": "kitten",
    "dog": "puppy"
}

print(baby_names)

{'cat': 'kitten', 'dog': 'puppy'}


In [3]:
print(list(baby_names.keys()))
print(list(baby_names.values()))
print(list(baby_names.items()))
print(baby_names.popitem())  # last inserted item

['cat', 'dog']
['kitten', 'puppy']
[('cat', 'kitten'), ('dog', 'puppy')]
('dog', 'puppy')


In [4]:
def my_func(**kwargs):
    for key, value in kwargs.items():
        print(f'{key} = {value}')

my_func(goose='gosling', kangaroo = 'joey')

goose = gosling
kangaroo = joey


In [4]:
class MyClass:
    def __init__(self):
        self.alligator = "hatchling"
        self.elephant = "calf"
        
a = MyClass()
for key, value in a.__dict__.items():
    print(f'{key} = {value}')

alligator = hatchling
elephant = calf


In [5]:
votes = {
"otter": 1281,
"polar bear": 587,
"fox": 863,
}

In [14]:
def populate_ranks(votes, ranks):
    names = list(votes.keys())
    names.sort(key=votes.get, reverse=True)
    for i, name in enumerate(names, 1):
        ranks[name] = i

In [15]:
def get_winner(ranks):
    return next(iter(ranks))

In [16]:
ranks = {}
populate_ranks(votes, ranks)
print(ranks)
winner = get_winner(ranks)
print(winner)

{'otter': 1, 'fox': 2, 'polar bear': 3}
otter


In [17]:
from collections.abc import MutableMapping

class SortedDict(MutableMapping):
    def __init__(self):
        super().__init__()
        self.data = {}
        
    def __getitem__(self, key):
        return self.data[key]
    
    def __setitem__(self, key, value):
        self.data[key] = value
        
    def __delitem__(self, key):
        del self.data[key]
        
    def __iter__(self):
        keys = list(self.data.keys())
        keys.sort()
        for key in keys:
            yield key
            
    def __len__(self):
        return len(self.data)

In [18]:
sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)
winner = get_winner(sorted_ranks)
print(winner)

{'otter': 1, 'fox': 2, 'polar bear': 3}
fox


In [19]:
def get_winner(ranks):
    for name, rank in ranks.items():
        if rank == 1:
            return name
        
winner = get_winner(sorted_ranks)
print(winner)

otter


In [20]:
def get_winner(ranks):
    if not isinstance(ranks, dict):
        raise TypeError("Must provide a dict instance")
    return next(iter(ranks))

get_winner(sorted_ranks)

TypeError: Must provide a dict instance

In [22]:
from typing import Dict, MutableMapping

def populate_ranks(votes: Dict[str, int],
                   ranks: Dict[str, int]) -> None:
    names = list(votes.keys())
    names.sort(key=votes.__getitem__, reverse=True)
    for i, name in enumerate(names, 1):
        ranks[name] = i

def get_winner(ranks: Dict[str, int]) -> str:
    return next(iter(ranks))

class SortedDict(MutableMapping[str, int]):
    def __init__(self):
        self._data: Dict[str, int] = {}

    def __getitem__(self, key: str) -> int:
        return self._data[key]

    def __setitem__(self, key: str, value: int) -> None:
        self._data[key] = value

    def __delitem__(self, key: str) -> None:
        del self._data[key]

    def __iter__(self):
        for key in sorted(self._data.keys()):
            yield key

    def __len__(self) -> int:
        return len(self._data)

    def items(self):
        for key in self:
            yield (key, self._data[key])


In [23]:
votes = {
    "otter": 1281,
    "polar bear": 587,
    "fox": 863,
}


In [24]:
sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)
winner = get_winner(sorted_ranks)
print(winner)

AttributeError: 'SortedDict' object has no attribute 'data'

## 26. Prefer `get` over `in` and `KeyError` to Handle Missing Dictionary Keys

In [36]:
counters = {
    "pumpernickel": 2,
    "sourdough": 1,
}

In [28]:
key = "wheat"

if key in counters:
    count = counters[key]
else:
    count = 0
    
counters[key] = count + 1

print(counters)

{'pumpernickel': 2, 'sourdough': 1, 'wheat': 2}


In [37]:
try:
    count = counters[key]
except KeyError:
    count = 0
    
counters[key] = count + 1

In [38]:
count = counters.get(key, 0)
counters[key] = count + 1

>**NOTE**
>
>If you’re maintaining dictionaries of counters like
>this, it’s worth considering the `Counter` class from
>the collections built-in module, which provides
>most of the functionality you're likely to need

In [41]:
votes = {
    "baguette": ['Bob','Alice'],
    'ciabatta': ['Coco','Deb']
}

key = 'brioche'
who = 'Elmer'

if key in votes:
    names = votes[key]
else:
    votes[key] = names = []
    
names.append(who)
print(votes)

{'baguette': ['Bob', 'Alice'], 'ciabatta': ['Coco', 'Deb'], 'brioche': ['Elmer']}


In [42]:
try:
    names = votes[key]
except KeyError:
    votes[key] = names = []
    
names.append(who)

In [43]:
names = votes.get(key)
if names is None:
    votes[key] = names = []
    
names.append(who)

In [44]:
if (names := votes.get(key)) is None:
    votes[key] = names = []
names.append(who)

In [47]:
# setdefault: jeśli 'key' istnieje w słowniku 'votes', zwraca istniejącą wartość (np. listę)
# jeśli 'key' nie istnieje, tworzy wpis votes[key] = [] (domyślna wartość) i zwraca tę nowo utworzoną listę
names = votes.setdefault(key, [])
names.append(who)
# końcowy efekt: votes[key] będzie listą zawierającą 'who'

In [48]:
data = {}
key = "foo"
value = []
data.setdefault(key, value)
print("Before:", data)
value.append("hello")
print("After: ", data)

Before: {'foo': []}
After:  {'foo': ['hello']}


## 27.  Prefer `defaultdict` over `setdefault` to Handle Missing Items in Internal State

In [50]:
visits = {
    "Mexico": {"Tulum", "Puerto Vallarta"},
    "Japan": {"Hekone"},
}

In [51]:
# Short
visits.setdefault("France", set()).add("Arles")

In [52]:
# Long
if (japan := visits.get("Japan")) is None:
    visits["Japan"] = japan = set()
    
japan.add("Kyoto")
print(visits)

{'Mexico': {'Tulum', 'Puerto Vallarta'}, 'Japan': {'Kyoto', 'Hekone'}, 'France': {'Arles'}}


In [55]:
class Visits:
    def __init__(self):
        self.data = {}
        
    def add(self, country, city):
        city_set = self.data.setdefault(country, set())
        city_set.add(city)

In [56]:
visits = Visits()
visits.add("Russia", "Yekaterinburg")
visits.add("Tanzania", "Zanzibar")
print(visits.data)

{'Russia': {'Yekaterinburg'}, 'Tanzania': {'Zanzibar'}}


In [57]:
from collections import defaultdict

class Visits:
    def __init__(self):
        self.data = defaultdict(set)
    
    def add(self, country, city):
        self.data[country].add(city)
        
visits = Visits()
visits.add("England", "Bath")
visits.add("England", "London")
print(visits.data)

defaultdict(<class 'set'>, {'England': {'London', 'Bath'}})
